In [1]:
import numpy as np
import tensorflow as tf
import pandas as pd
import keras_hub
import os
from tensorflow.keras.callbacks import ModelCheckpoint, CSVLogger
from sklearn import metrics
from pathlib import Path
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.layers import Dense, Dropout

C:\Users\dorrr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Data Preparation

In [2]:
# actor splits
splits = pd.read_csv('actor_splits.csv')
splits['actor_id'] = splits['actor_id'].astype(int)
splits['split'] = splits['split'].str.strip().str.lower()
actor_split = dict(zip(splits['actor_id'], splits['split']))

In [3]:
def get_label_actor(filename):

    name = os.path.splitext(filename)[0]
    parts = name.split('-')

    if len(parts) != 6:
        raise ValueError(f'Unexpected filename: {filename}')

    emotion = int(parts[1])
    actor_id = int(parts[-1])

    y = emotion - 1

    return y, actor_id

In [4]:
X_train, y_train, id_train = [], [], []
X_val, y_val, id_val = [], [], []
X_test, y_test, id_test = [], [], []

files = sorted([f for f in os.listdir('faces') if f.endswith('.npz')])

for filename in files:

    # Get label and actor ID
    y, actor_id = get_label_actor(filename)
    sample_id = os.path.splitext(filename)[0]

    # Load facial frames
    with np.load(os.path.join('faces', filename)) as data:
        X = data['frames']

    # Check frame shape
    if X.shape != (16, 224, 224, 3):
        raise ValueError(f'Invalid shape: {filename}, {X.shape}')

    # Actor-independent split
    split = actor_split[actor_id]

    if split == 'train':
        X_train.append(X)
        y_train.append(y)
        id_train.append(sample_id)

    elif split == 'val':
        X_val.append(X)
        y_val.append(y)
        id_val.append(sample_id)

    elif split == 'test':
        X_test.append(X)
        y_test.append(y)
        id_test.append(sample_id)

    else:
        raise ValueError(f'Unknown split: {split}')

In [5]:
X_train = np.array(X_train, dtype=np.uint8)
X_val = np.array(X_val, dtype=np.uint8)
X_test = np.array(X_test, dtype=np.uint8)

y_train = np.array(y_train, dtype=np.int32)
y_val = np.array(y_val, dtype=np.int32)
y_test = np.array(y_test, dtype=np.int32)

In [6]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (960, 16, 224, 224, 3)
y_train: (960,)
X_val: (240, 16, 224, 224, 3)
y_val: (240,)
X_test: (240, 16, 224, 224, 3)
y_test: (240,)


In [7]:
IMG_SIZE = 224
NUM_FRAMES = 16
NUM_CLASSES = 8

## 2. Resnet18 + Mean Pooling model

In [8]:
resnet18 = keras_hub.models.ResNetBackbone.from_preset(
    "resnet_18_imagenet"
)

resnet18.trainable = False

In [9]:
model = keras.Sequential()

model.add(keras.Input(shape=(NUM_FRAMES, IMG_SIZE, IMG_SIZE, 3)))

model.add(layers.TimeDistributed(
    keras.Sequential([resnet18, layers.GlobalAveragePooling2D()]), 
    name='sequence_features'
))

model.add(layers.GlobalAveragePooling1D(name='pooled_features'))

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(NUM_CLASSES, activation='softmax'))

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequence_features               │ (None, 16, 512)        │    11,186,112 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pooled_features                 │ (None, 512)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │         1,032 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,252,808 (42.93 MB)

 Trainable params: 66,696 (260.53 KB)

 Non-trainable params: 11,186,112 (42.67 MB)

In [10]:
# One-hot encoding
y_train_encoded = keras.utils.to_categorical(y_train, 8)
y_val_encoded = keras.utils.to_categorical(y_val, 8)
y_test_encoded = keras.utils.to_categorical(y_test, 8)

# Set up optimizer
optmz = tf.keras.optimizers.RMSprop(learning_rate=0.0001)
model.compile(
    loss='categorical_crossentropy', 
    optimizer=optmz, 
    metrics=[keras.metrics.F1Score(average='macro', name='macro_f1')]
)

# create checkpoints to save model during training and save training data into csv
modelname = 'resnet18_mean'
filepath = modelname + '.keras'
checkpoint = keras.callbacks.ModelCheckpoint(
    'resnet18_mean.keras', 
    monitor='val_macro_f1', 
    verbose=1, 
    save_best_only=True, 
    mode='max'
)
csv_logger = CSVLogger(modelname + '.csv')
callbacks_list = [checkpoint, csv_logger]

In [11]:
# model training
history = model.fit(
    X_train, 
    y_train_encoded, 
    batch_size=2, 
    epochs=10, 
    validation_data=(X_val, y_val_encoded), 
    callbacks=callbacks_list
)

print("Validation Macro-F1:", max(history.history['val_macro_f1']))

Epoch 1/10
480/480 ━━━━━━━━━━━━━━━━━━━━ 0s 412ms/step - loss: 2.1996 - macro_f1: 0.1394
Epoch 1: val_macro_f1 improved from None to 0.13525, saving model to resnet18_mean.keras

Epoch 1: finished saving model to resnet18_mean.keras
480/480 ━━━━━━━━━━━━━━━━━━━━ 262s 523ms/step - loss: 2.1996 - macro_f1: 0.1394 - val_loss: 2.0529 - val_macro_f1: 0.1353
Epoch 2/10
480/480 ━━━━━━━━━━━━━━━━━━━━ 0s 415ms/step - loss: 2.1220 - macro_f1: 0.1365
Epoch 2: val_macro_f1 did not improve from 0.13525
480/480 ━━━━━━━━━━━━━━━━━━━━ 249s 518ms/step - loss: 2.1220 - macro_f1: 0.1365 - val_loss: 2.0430 - val_macro_f1: 0.0815
Epoch 3/10
480/480 ━━━━━━━━━━━━━━━━━━━━ 0s 416ms/step - loss: 2.0636 - macro_f1: 0.1554
Epoch 3: val_macro_f1 did not improve from 0.13525
480/480 ━━━━━━━━━━━━━━━━━━━━ 249s 520ms/step - loss: 2.0636 - macro_f1: 0.1554 - val_loss: 2.0067 - val_macro_f1: 0.1066
Epoch 4/10
480/480 ━━━━━━━━━━━━━━━━━━━━ 0s 416ms/step - loss: 2.0234 - macro_f1: 0.1801
Epoch 4: val_macro_f1 did not improve f

In [12]:
# Feature extraction model
model = keras.models.load_model(filepath)

inputs = keras.Input(shape=(16, 224, 224, 3))

x = inputs
outputs = []

for layer in model.layers:

    x = layer(x)

    if layer.name == 'sequence_features':
        seq = x

    if layer.name == 'pooled_features':
        pool = x

# Final output
prob = x

feature_model = keras.Model(
    inputs=inputs,
    outputs=[seq, pool, prob]
)

C:\Users\dorrr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 66 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(store)


In [13]:
# Train features
train_seq, train_pool, train_prob = feature_model.predict(
    X_train, batch_size=2
)

# Validation features
val_seq, val_pool, val_prob = feature_model.predict(
    X_val, batch_size=2
)

# Test features
test_seq, test_pool, test_prob = feature_model.predict(
    X_test, batch_size=2
)

# Combine features
sequence_features = np.concatenate(
    [train_seq, val_seq, test_seq], axis=0
)

pooled_features = np.concatenate(
    [train_pool, val_pool, test_pool], axis=0
)

probabilities = np.concatenate(
    [train_prob, val_prob, test_prob], axis=0
)

# Combine metadata
sample_id = np.array(id_train + id_val + id_test)

label = np.concatenate([y_train, y_val, y_test])

split = np.array(
    ['train'] * len(id_train) +
    ['val'] * len(id_val) +
    ['test'] * len(id_test)
)

480/480 ━━━━━━━━━━━━━━━━━━━━ 195s 400ms/step
120/120 ━━━━━━━━━━━━━━━━━━━━ 49s 411ms/step
120/120 ━━━━━━━━━━━━━━━━━━━━ 50s 413ms/step


In [14]:
np.savez_compressed(
    'resnet18_mean_features.npz',
    sample_id=sample_id,
    label=label,
    split=split,
    sequence_features=sequence_features,
    pooled_features=pooled_features,
    probabilities=probabilities
)